# 02d — Train EfficientNetV2-M

Anggota 4 — Kaggle GPU. Scaled mobile-conv CNN; memakai statistik normalisasi inception 0.5/0.5, otomatis ditangani oleh timm.data.resolve_model_data_config di skeleton bersama (jangan pernah hardcode mean/std ImageNet di sini).

Notebook ini byte-identical dengan ketiga file `02x_Train_*.ipynb` lainnya di bawah sel `CONFIG`
(PRD §5.1) — lihat `SKELETON_VERSION` di dalamnya untuk stempel versi logika bersama.


In [ ]:
# CONFIG — satu-satunya sel yang boleh berbeda di 02a-02d (PRD §5.1).
# Isi yang diizinkan: ARCH_NAME, TIMM_CHECKPOINT_TAG, override hyperparameter
# per-arsitektur (LR, drop_path_rate, batch size / grad-accum), slug checkpoint-dataset,
# dan adaptasi resolusi yang diwajibkan §7.7. Selain itu tidak boleh berbeda.
CONFIG = {
    "ARCH_NAME": 'tf_efficientnetv2_m',
    "TIMM_CHECKPOINT_TAG": 'tf_efficientnetv2_m.in21k_ft_in1k',
    "DROP_PATH_RATE": 0.2,
    "STAGE_RESOLUTIONS": {1: 224, 2: 384, 3: 512},  # {1: stage1, 2: stage2, 3: stage3} px
    "STAGE_LR": {1: 0.001, 2: 0.0001, 3: 1e-05},
    "STAGE_BATCH_SIZE": {1: 32, 2: 24, 3: 12},
    "STAGE_GRAD_ACCUM": {1: 1, 2: 2, 3: 3},
    "CHECKPOINT_DATASET_SLUG": 'bdc2026-ckpt-effnetv2',
    "MASTER_DATASET_VERSION": 1,  # dipin setelah NB01 publish (PRD §6.3)
}


# 02x — Training Engine (skeleton bersama)

Notebook 02a-02d byte-identical kecuali sel `CONFIG` di atas (PRD §5.1). Perubahan apa pun pada
logika bersama di bawah ini WAJIB menaikkan `SKELETON_VERSION` dan didistribusikan ulang ke
keempat anggota tim.

Alur per fold: Stage 1 (linear probe) -> Stage 2 (full fine-tune, early-stopping eligible) ->
Stage 3 (refinement) -> ekspor OOF. Sepenuhnya resumable lintas restart sesi Kaggle (PRD §9).


In [ ]:
# Catatan (deviasi dari PRD §12 yang secara literal minta exact-pin semua dependency):
# image GPU Kaggle sudah membawa torch/torchvision versi yang cocok dengan driver CUDA-nya, plus
# scikit-learn/pandas/numpy/tqdm yang dipakai puluhan paket bawaan lain — memaksa versi lama persis
# di sini bisa merusak link CUDA atau bentrok dengan resolver pip di seluruh environment. Hanya
# `timm` (yang tidak bawaan Kaggle) yang di-pin persis; sisanya memakai versi bawaan platform,
# tercatat lewat print(torch.__version__ / timm.__version__) di sel berikutnya untuk audit trail.
!pip install -q timm==1.0.11


In [ ]:
import hashlib
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import timm.data
import timm.loss
import timm.utils
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True  # determinisme bitwise tidak wajib; semua seed sudah dicatat (PRD 5.4)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 3
print("device:", DEVICE, "| torch:", torch.__version__, "| timm:", timm.__version__)


In [ ]:
SKELETON_VERSION = "skeleton-v1.1.0"  # naikkan versi ini untuk SETIAP perubahan logika bersama; distribusikan ke 4 anggota

MASTER_DATASET_SLUG = "bdc2026-master-data"  # tetap sama untuk seluruh tim; hanya VERSION yang dipin per CONFIG
MASTER_DATASET_MOUNT = Path(f"/kaggle/input/{MASTER_DATASET_SLUG}")
PROCESSED_ROOT = str(MASTER_DATASET_MOUNT / "processed")  # satu konstanta string bersama (PRD 8)
FOLD_ASSIGNMENT_PATH = MASTER_DATASET_MOUNT / "manifests" / "fold_assignment.csv"

# Gagal cepat & jelas kalau dataset tidak ke-mount, daripada FileNotFoundError samar setelah
# beberapa menit pip install. Penyebab paling umum: dataset gagal ter-attach ke kernel ini
# (kadang terjadi setelah error "ConcurrencyViolation / Failed to save draft" saat import
# notebook -- attachment dataset-nya tidak ikut ke-commit). Perbaikannya: cek panel "Input" di
# kanan, klik "Add Input" kalau bdc2026-master-data hilang, lalu "Save Version" ulang.
if not FOLD_ASSIGNMENT_PATH.exists():
    available = sorted(p.name for p in Path("/kaggle/input").iterdir()) if Path("/kaggle/input").exists() else []
    raise FileNotFoundError(
        f"Dataset '{MASTER_DATASET_SLUG}' tidak ketemu di {FOLD_ASSIGNMENT_PATH}.\n"
        f"Isi /kaggle/input/ saat ini: {available}\n"
        f"Cek panel 'Input' di kanan notebook -- kalau '{MASTER_DATASET_SLUG}' hilang, klik "
        f"'Add Input' untuk attach ulang, lalu 'Save Version' lagi."
    )

WORKING_ROOT = Path("/kaggle/working")
CKPT_ROOT = WORKING_ROOT / "checkpoints" / CONFIG["ARCH_NAME"]
OOF_ROOT = WORKING_ROOT / "oof"
CKPT_ROOT.mkdir(parents=True, exist_ok=True)
OOF_ROOT.mkdir(parents=True, exist_ok=True)


def file_md5(path, chunk_size=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def hash_config(cfg: dict) -> str:
    return hashlib.sha256(json.dumps(cfg, sort_keys=True, default=str).encode()).hexdigest()[:16]


def build_provenance(cfg: dict) -> dict:
    # PRD 5.2 — setiap checkpoint dict DAN setiap sidecar CSV OOF wajib membawa 4 stempel ini.
    return {
        "fold_assignment_md5": file_md5(FOLD_ASSIGNMENT_PATH),
        "master_dataset_version": cfg["MASTER_DATASET_VERSION"],
        "skeleton_version": SKELETON_VERSION,
        "config_hash": hash_config(cfg),
    }


PROVENANCE = build_provenance(CONFIG)
print("provenance:", PROVENANCE)


## Dataset & transform (PRD §7.3, §8)

Identitas numerik kanonis saja: `image_ids`/`labels` adalah array `np.int64` bertipe tetap, path
dibangun ulang dari integer + satu konstanta string bersama. Tidak ada objek string Python
per-gambar yang pernah disimpan (menghindari ledakan RAM fork/copy-on-write yang jadi alasan desain ini).


In [ ]:
class WasteDataset(Dataset):
    def __init__(self, image_ids: np.ndarray, labels, transform, return_id: bool = False):
        self.image_ids = image_ids.astype(np.int64)
        self.labels = labels.astype(np.int64) if labels is not None else None
        self.transform = transform
        self.return_id = return_id

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = int(self.image_ids[idx])
        path = f"{PROCESSED_ROOT}/{image_id}.jpg"  # hanya integer + satu konstanta bersama (PRD 8)
        with Image.open(path) as im:
            im = im.convert("RGB")
            tensor = self.transform(im)
        if self.return_id:
            return tensor, image_id
        return tensor, int(self.labels[idx])


def build_train_transform(img_size, mean, std, erasing_p, color_jitter_scale):
    return T.Compose([
        T.RandomResizedCrop(img_size, scale=(0.30, 1.0), ratio=(0.75, 1.333),
                             interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(p=0.5),
        T.ColorJitter(brightness=0.4 * color_jitter_scale, contrast=0.4 * color_jitter_scale,
                      saturation=0.3 * color_jitter_scale, hue=0.05 * color_jitter_scale),
        T.RandomGrayscale(p=0.15),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
        T.RandomErasing(p=erasing_p, scale=(0.02, 0.2), value="random"),
    ])


def build_eval_transform(img_size, mean, std):
    resize_short_side = round(img_size / 0.95)
    return T.Compose([
        T.Resize(resize_short_side, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(img_size),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])


TTA_SPEC_VERSION = "identity+hflip-softmax-mean-v1"  # wajib byte-identical dengan salinannya di NB04 (PRD 7.3, 7.8, 11)


@torch.no_grad()
def tta_predict_probs(model, images: torch.Tensor) -> torch.Tensor:
    """TTA 2 view: {identity, horizontal flip}, rata-rata post-softmax. Hanya untuk OOF/NB04, tidak untuk validasi per-epoch."""
    images = images.to(DEVICE, non_blocking=True)
    probs_identity = F.softmax(model(images), dim=1)
    probs_flip = F.softmax(model(torch.flip(images, dims=[3])), dim=1)
    return (probs_identity + probs_flip) / 2.0


def build_weighted_sampler(labels: np.ndarray) -> WeightedRandomSampler:
    class_counts = np.bincount(labels, minlength=NUM_CLASSES)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[labels]
    return WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(labels),
        replacement=True,  # PRD 7.2 — sampler-level saja, tidak pernah class weight di dalam loss
    )


def make_loader(dataset, sampler=None, shuffle=False, batch_size=32, drop_last=False):
    return DataLoader(
        dataset, batch_size=batch_size, sampler=sampler, shuffle=(shuffle and sampler is None),
        drop_last=drop_last, num_workers=2, persistent_workers=True, pin_memory=True, prefetch_factor=2,
    )


## Pembuatan model (PRD §7.1) & freeze head (PRD §7.5 Stage 1)

Statistik normalisasi diresolve per checkpoint — tidak pernah hardcode mean/std ImageNet.


In [ ]:
def create_model(timm_tag: str, drop_path_rate: float, img_size: int) -> nn.Module:
    kwargs = dict(pretrained=True, num_classes=NUM_CLASSES, drop_path_rate=drop_path_rate, img_size=img_size)
    try:
        model = timm.create_model(timm_tag, **kwargs)  # TRANSFER LEARNING OCCURS HERE: ImageNet-pretrained weights are loaded BEFORE any competition image is seen. The 3-class head is randomly initialized.
    except TypeError:
        kwargs.pop("img_size")  # sebagian arsitektur (mis. keluarga fully-convolutional) menolak img_size saat konstruksi
        model = timm.create_model(timm_tag, **kwargs)  # TRANSFER LEARNING OCCURS HERE: ImageNet-pretrained weights are loaded BEFORE any competition image is seen. The 3-class head is randomly initialized.
    return model


def resolve_norm_stats(model: nn.Module):
    data_cfg = timm.data.resolve_model_data_config(model)
    return list(data_cfg["mean"]), list(data_cfg["std"])


def freeze_backbone_except_head(model: nn.Module) -> nn.Module:
    for p in model.parameters():
        p.requires_grad = False
    classifier = model.get_classifier()
    for p in classifier.parameters():
        p.requires_grad = True
    return classifier


def unfreeze_all(model: nn.Module):
    for p in model.parameters():
        p.requires_grad = True


## §7.7 Resolution smoke-test gate (SEBELUM run panjang mana pun)

Forward satu dummy batch di resolusi 224/384/512, termasuk konstruksi model di tiap `img_size`.
Jika suatu resolusi gagal untuk arsitektur ini, adaptasinya dibatasi hanya di
`CONFIG["STAGE_RESOLUTIONS"]` (dicatat di bawah) — konsistensi dalam satu arsitektur lebih penting
daripada memaksakan 512.


In [ ]:
def resolution_smoke_test(timm_tag: str, drop_path_rate: float, resolutions=(224, 384, 512)):
    report = {}
    for r in resolutions:
        try:
            m = create_model(timm_tag, drop_path_rate, r).to(DEVICE).eval()
            dummy = torch.randn(2, 3, r, r, device=DEVICE)
            with torch.no_grad():
                out = m(dummy)
            assert out.shape == (2, NUM_CLASSES)
            report[r] = {"ok": True}
            del m, dummy, out
            torch.cuda.empty_cache()
        except Exception as e:
            report[r] = {"ok": False, "error": str(e)}
        print(f"resolusi {r}: {'OK' if report[r]['ok'] else 'GAGAL - ' + report[r]['error']}")
    return report


_gate_report = resolution_smoke_test(CONFIG["TIMM_CHECKPOINT_TAG"], CONFIG["DROP_PATH_RATE"])
with open(CKPT_ROOT / "resolution_gate_report.json", "w") as f:
    json.dump(_gate_report, f, indent=2)

for _stage, _res in CONFIG["STAGE_RESOLUTIONS"].items():
    assert _gate_report[_res]["ok"], (
        f"Resolusi stage {_stage} ({_res}px) gagal smoke test untuk {CONFIG['ARCH_NAME']}. "
        f"Adaptasi CONFIG['STAGE_RESOLUTIONS'] ke set resolusi terdekat yang didukung dan catat perubahannya (PRD 7.7)."
    )


## Definisi stage (PRD §7.5) & jadwal LR

`current_stage` dan `global_step` adalah state utama yang di-reset per stage (PRD §9.1) —
`global_step` menghitung jumlah optimizer step sejak stage saat ini dimulai, dan jadwal LR adalah
fungsi murni dari nilai itu.


In [ ]:
STAGE_DEFS = {
    1: dict(name="warmup_linear_probe", epochs=3, lr_key="lr1", schedule="constant",
            loss="ce_ls", mixup=False, erasing_p=0.25, color_jitter_scale=1.0,
            optimizer_scope="head", early_stop_eligible=False),
    2: dict(name="full_finetune", epochs=12, lr_key="lr2", schedule="warmup_cosine",
            loss="soft_ce", mixup=True, erasing_p=0.25, color_jitter_scale=1.0,
            optimizer_scope="all", early_stop_eligible=True),
    3: dict(name="refinement", epochs=4, lr_key="lr3", schedule="cosine",
            loss="ce_ls", mixup=False, erasing_p=0.10, color_jitter_scale=0.5,
            optimizer_scope="all", early_stop_eligible=False),
}
STAGE_MIN_LR = {1: None, 2: 1e-6, 3: 1e-7}  # jadwal constant (stage 1) tidak punya floor
WARMUP_EPOCHS_STAGE2 = 1
EARLY_STOP_PATIENCE = 4
EARLY_STOP_MIN_EPOCHS = WARMUP_EPOCHS_STAGE2 + 3  # "minimum 3 epoch harus berlalu pasca-warmup" (PRD 15)
GRAD_CLIP_NORM = 1.0
MID_EPOCH_CKPT_EVERY_STEPS = 500


def build_lr_lambda(schedule: str, warmup_steps: int, total_steps: int, min_lr: float, base_lr: float):
    if schedule == "constant":
        return lambda step: 1.0

    def lr_lambda(step):
        if schedule == "warmup_cosine" and step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        anchor = warmup_steps if schedule == "warmup_cosine" else 0
        progress = (step - anchor) / max(1, total_steps - anchor)
        progress = min(max(progress, 0.0), 1.0)
        cosine = 0.5 * (1 + math.cos(math.pi * progress))
        min_ratio = min_lr / base_lr
        return min_ratio + (1 - min_ratio) * cosine

    return lr_lambda


def build_optimizer_and_scheduler(model: nn.Module, stage: int, cfg: dict, steps_per_epoch: int):
    stage_def = STAGE_DEFS[stage]
    base_lr = cfg["STAGE_LR"][stage]
    if stage_def["optimizer_scope"] == "head":
        params = model.get_classifier().parameters()
    else:
        params = model.parameters()
    optimizer = torch.optim.AdamW(params, lr=base_lr, weight_decay=0.05)  # optimizer baru di setiap batas stage (PRD 7.5)

    total_steps = stage_def["epochs"] * steps_per_epoch
    warmup_steps = WARMUP_EPOCHS_STAGE2 * steps_per_epoch if stage_def["schedule"] == "warmup_cosine" else 0
    lr_lambda = build_lr_lambda(stage_def["schedule"], warmup_steps, total_steps, STAGE_MIN_LR[stage], base_lr)
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


def build_loss_and_mixup(stage: int):
    stage_def = STAGE_DEFS[stage]
    if stage_def["mixup"]:
        mixup_fn = timm.data.Mixup(
            mixup_alpha=0.2, cutmix_alpha=1.0, prob=0.8, switch_prob=0.5,
            label_smoothing=0.1, num_classes=NUM_CLASSES,
        )
        loss_fn = timm.loss.SoftTargetCrossEntropy()
    else:
        mixup_fn = None
        loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    return loss_fn, mixup_fn


## Checkpointing & resume (PRD §9)

`epoch=-1, global_step=0` adalah sentinel yang ditulis tepat saat batas stage terlewati: bobot
model/EMA dibawa terus, tapi belum ada state optimizer/scheduler untuk stage baru yang perlu dipulihkan.


In [ ]:
def optimizer_to_device(optimizer, device):
    for state in optimizer.state.values():
        for k, v in state.items():
            if torch.is_tensor(v):
                state[k] = v.to(device)


def atomic_save(state: dict, path: Path):
    tmp_path = str(path) + ".tmp"
    torch.save(state, tmp_path)
    os.replace(tmp_path, path)  # PRD 9.2 — atomic write


def make_checkpoint(model, ema, optimizer, scheduler, scaler, epoch, global_step, current_stage, fold,
                     best_val_macro_f1, best_weights_variant, epochs_since_improvement, cfg, fold_complete=False):
    return {
        "model_state": model.state_dict(),
        "ema_state": ema.module.state_dict() if ema is not None else None,
        "optimizer_state": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "scaler_state": scaler.state_dict() if scaler is not None else None,
        "epoch": epoch,
        "global_step": global_step,
        "current_stage": current_stage,
        "fold": fold,
        "best_val_macro_f1": best_val_macro_f1,
        "best_weights_variant": best_weights_variant,
        "epochs_since_improvement": epochs_since_improvement,
        "config_snapshot": {**cfg, "provenance": PROVENANCE},
        "fold_complete": fold_complete,
    }


def fold_dir(fold: int) -> Path:
    d = CKPT_ROOT / f"fold{fold}"
    d.mkdir(parents=True, exist_ok=True)
    return d


def load_last_checkpoint(fold: int):
    path = fold_dir(fold) / "last.ckpt"
    if not path.exists():
        return None
    return torch.load(path, map_location="cpu")


## Fungsi train / validate per epoch (PRD §7.6, §7.8)


In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler, loss_fn, mixup_fn,
                     ema, global_step, grad_accum_steps, mid_epoch_ckpt_fn):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    for step, (images, targets) in enumerate(tqdm(loader, desc="train", leave=False)):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        if mixup_fn is not None:
            images, targets = mixup_fn(images, targets)

        with torch.cuda.amp.autocast():
            logits = model(images)
            loss = loss_fn(logits, targets) / grad_accum_steps
        scaler.scale(loss).backward()
        running_loss += loss.item() * grad_accum_steps

        is_last_micro_batch = (step + 1) == len(loader)
        if (step + 1) % grad_accum_steps == 0 or is_last_micro_batch:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            if ema is not None:
                ema.update(model)
            global_step += 1
            if global_step % MID_EPOCH_CKPT_EVERY_STEPS == 0:
                mid_epoch_ckpt_fn(global_step)

    return running_loss / len(loader), global_step


@torch.no_grad()
def validate_epoch(model, ema, loader):
    # WAJIB: eval() + no_grad supaya dropout/BN tidak aktif saat validasi (PRD 7.6, 14.9)
    eval_transform_loader = loader  # sudah dibangun dengan EVAL transform, tanpa TTA (PRD 7.6)

    def run(m):
        m.eval()
        all_preds, all_targets = [], []
        for images, targets in eval_transform_loader:
            images = images.to(DEVICE, non_blocking=True)
            logits = m(images)
            preds = logits.argmax(dim=1).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(targets.numpy())
        y_pred = np.concatenate(all_preds)
        y_true = np.concatenate(all_targets)
        macro_f1 = f1_score(y_true, y_pred, average="macro")
        per_class_f1 = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2]).tolist()
        return macro_f1, per_class_f1

    raw_f1, raw_per_class = run(model)
    ema_f1, ema_per_class = (None, None)
    if ema is not None:
        ema_f1, ema_per_class = run(ema.module)
    return {
        "raw_macro_f1": raw_f1, "raw_per_class_f1": raw_per_class,
        "ema_macro_f1": ema_f1, "ema_per_class_f1": ema_per_class,
    }


## Orkestrasi 3-stage per fold, dengan resume (PRD §7.5, §9.3, §15)


In [ ]:
def append_training_log(fold, row: dict):
    log_path = fold_dir(fold) / "training_log.csv"
    df_row = pd.DataFrame([row])
    if log_path.exists():
        df_row.to_csv(log_path, mode="a", header=False, index=False)
    else:
        df_row.to_csv(log_path, mode="w", header=True, index=False)


def build_fold_datasets(fold: int, img_size: int, erasing_p: float, color_jitter_scale: float, mean, std):
    fa = pd.read_csv(FOLD_ASSIGNMENT_PATH)
    train_rows = fa[fa["fold"] != fold]
    val_rows = fa[fa["fold"] == fold]
    train_tf = build_train_transform(img_size, mean, std, erasing_p, color_jitter_scale)
    eval_tf = build_eval_transform(img_size, mean, std)
    train_ds = WasteDataset(train_rows["image_id"].values, train_rows["label"].values, train_tf)
    val_ds = WasteDataset(val_rows["image_id"].values, val_rows["label"].values, eval_tf)
    sampler = build_weighted_sampler(train_rows["label"].values)
    return train_ds, val_ds, sampler


def run_fold(fold: int, cfg: dict):
    print(f"\n{'=' * 20} {cfg['ARCH_NAME']} | fold {fold} {'=' * 20}")
    resume_ckpt = load_last_checkpoint(fold)

    if resume_ckpt is not None and resume_ckpt.get("fold_complete"):
        print(f"fold {fold} sudah selesai — lanjut ke pengecekan OOF")
        return

    start_stage = resume_ckpt["current_stage"] if resume_ckpt is not None else 1
    img_size = cfg["STAGE_RESOLUTIONS"][start_stage]
    model = create_model(cfg["TIMM_CHECKPOINT_TAG"], cfg["DROP_PATH_RATE"], img_size).to(DEVICE)
    mean, std = resolve_norm_stats(model)

    ema = None
    best_val_macro_f1 = -math.inf
    best_weights_variant = None
    epochs_since_improvement = 0

    if resume_ckpt is not None:
        model.load_state_dict(resume_ckpt["model_state"])
        best_val_macro_f1 = resume_ckpt["best_val_macro_f1"]
        best_weights_variant = resume_ckpt["best_weights_variant"]
        epochs_since_improvement = resume_ckpt["epochs_since_improvement"]
        if resume_ckpt["ema_state"] is not None:
            ema = timm.utils.ModelEmaV2(model, decay=0.9995)
            ema.module.load_state_dict(resume_ckpt["ema_state"])

    for stage in range(start_stage, 4):
        stage_def = STAGE_DEFS[stage]
        img_size = cfg["STAGE_RESOLUTIONS"][stage]

        resuming_within_this_stage = (
            resume_ckpt is not None and resume_ckpt["current_stage"] == stage and resume_ckpt["epoch"] >= 0
        )

        if stage != start_stage:
            resume_ckpt = None  # batas stage sudah dilewati dalam proses ini; tidak ada lagi yang perlu dipulihkan dari disk

        if stage == 2 and ema is None:
            ema = timm.utils.ModelEmaV2(model, decay=0.9995)  # EMA diinisialisasi saat Stage 2 mulai (PRD 7.5)
            best_val_macro_f1 = -math.inf  # best.ckpt dilacak HANYA di Stage 2-3 (PRD 7.6)
            best_weights_variant = None
            epochs_since_improvement = 0

        if stage_def["optimizer_scope"] == "head":
            freeze_backbone_except_head(model)
        else:
            unfreeze_all(model)

        train_ds, val_ds, sampler = build_fold_datasets(
            fold, img_size, stage_def["erasing_p"], stage_def["color_jitter_scale"], mean, std,
        )
        batch_size = cfg["STAGE_BATCH_SIZE"][stage]
        grad_accum = cfg["STAGE_GRAD_ACCUM"][stage]
        train_loader = make_loader(train_ds, sampler=sampler, batch_size=batch_size, drop_last=True)
        val_loader = make_loader(val_ds, shuffle=False, batch_size=batch_size, drop_last=False)

        optimizer, scheduler = build_optimizer_and_scheduler(model, stage, cfg, steps_per_epoch=len(train_loader))
        scaler = torch.cuda.amp.GradScaler()
        loss_fn, mixup_fn = build_loss_and_mixup(stage)

        start_epoch = 0
        global_step = 0
        if resuming_within_this_stage:
            model.to(DEVICE)  # pindahkan ke GPU sebelum membuat/memulihkan optimizer (PRD 9.3 langkah 3)
            optimizer.load_state_dict(resume_ckpt["optimizer_state"])
            optimizer_to_device(optimizer, DEVICE)
            scheduler.load_state_dict(resume_ckpt["scheduler_state"])
            scaler.load_state_dict(resume_ckpt["scaler_state"])  # wajib, tidak opsional (PRD 9.3 langkah 5)
            start_epoch = resume_ckpt["epoch"] + 1  # counter epoch tidak boleh dobel-increment (PRD 9.3 langkah 7)
            global_step = resume_ckpt["global_step"]

        print(f"-- stage {stage} ({stage_def['name']}) @ {img_size}px, "
              f"epoch {start_epoch}->{stage_def['epochs'] - 1} --")

        for epoch in range(start_epoch, stage_def["epochs"]):
            def mid_epoch_ckpt_fn(gs, _epoch=epoch, _stage=stage):
                ckpt = make_checkpoint(model, ema, optimizer, scheduler, scaler, _epoch, gs, _stage, fold,
                                        best_val_macro_f1, best_weights_variant, epochs_since_improvement, cfg)
                atomic_save(ckpt, fold_dir(fold) / "last.ckpt")

            train_loss, global_step = train_one_epoch(
                model, train_loader, optimizer, scheduler, scaler, loss_fn, mixup_fn, ema,
                global_step, grad_accum, mid_epoch_ckpt_fn,
            )
            metrics = validate_epoch(model, ema, val_loader)
            candidates = [("raw", metrics["raw_macro_f1"])]
            if metrics["ema_macro_f1"] is not None:
                candidates.append(("ema", metrics["ema_macro_f1"]))
            epoch_best_variant, epoch_best_f1 = max(candidates, key=lambda t: t[1])

            append_training_log(fold, {
                "stage": stage, "epoch": epoch, "train_loss": train_loss,
                "val_macro_f1_raw": metrics["raw_macro_f1"], "val_macro_f1_ema": metrics["ema_macro_f1"],
                "lr": scheduler.get_last_lr()[0], "timestamp": time.time(),
            })
            print(f"   epoch {epoch}: loss={train_loss:.4f} raw_f1={metrics['raw_macro_f1']:.4f} "
                  f"ema_f1={metrics['ema_macro_f1']}")

            improved = stage != 1 and epoch_best_f1 > best_val_macro_f1
            if stage == 1:
                # Metrik Stage 1 adalah bukti untuk juri, bukan kandidat best.ckpt (PRD 7.6)
                pass
            elif improved:
                best_val_macro_f1 = epoch_best_f1
                best_weights_variant = epoch_best_variant
                epochs_since_improvement = 0
                best_state = model.state_dict() if epoch_best_variant == "raw" else ema.module.state_dict()
                atomic_save({"weights": best_state, "variant": epoch_best_variant,
                             "val_macro_f1": best_val_macro_f1, "stage": stage, "epoch": epoch,
                             "img_size": img_size, "provenance": PROVENANCE},
                            fold_dir(fold) / "best.ckpt")
            else:
                epochs_since_improvement += 1

            last_ckpt = make_checkpoint(model, ema, optimizer, scheduler, scaler, epoch, global_step, stage, fold,
                                         best_val_macro_f1, best_weights_variant, epochs_since_improvement, cfg)
            atomic_save(last_ckpt, fold_dir(fold) / "last.ckpt")

            if stage == 1 and epoch == stage_def["epochs"] - 1:
                lp_f1, lp_per_class = metrics["raw_macro_f1"], metrics["raw_per_class_f1"]
                with open(fold_dir(fold) / "linear_probe_evidence.json", "w") as f:
                    json.dump({"arch": cfg["ARCH_NAME"], "fold": fold, "macro_f1": lp_f1,
                               "per_class_f1": lp_per_class}, f, indent=2)

            if (stage_def["early_stop_eligible"] and epoch + 1 >= EARLY_STOP_MIN_EPOCHS
                    and epochs_since_improvement >= EARLY_STOP_PATIENCE):
                print(f"   early stop: stage {stage} epoch {epoch} "
                      f"({epochs_since_improvement} epoch tanpa perbaikan)")
                break

        # batas stage: simpan state sentinel segar untuk stage berikutnya (epoch=-1, global_step=0)
        next_stage = stage + 1
        if next_stage <= 3:
            sentinel = make_checkpoint(model, ema, None, None, None, -1, 0, next_stage, fold,
                                        best_val_macro_f1, best_weights_variant, epochs_since_improvement, cfg)
            atomic_save(sentinel, fold_dir(fold) / "last.ckpt")
        else:
            final = make_checkpoint(model, ema, None, None, None, stage_def["epochs"] - 1, global_step, stage, fold,
                                     best_val_macro_f1, best_weights_variant, epochs_since_improvement, cfg,
                                     fold_complete=True)
            atomic_save(final, fold_dir(fold) / "last.ckpt")

    print(f"fold {fold} training selesai. best_val_macro_f1={best_val_macro_f1:.4f} "
          f"(variant={best_weights_variant})")


## Push checkpoint lintas sesi (PRD §9.2)

Best-effort: push `checkpoints/{arch}/` ke Kaggle Dataset privat milik masing-masing anggota
`bdc2026-ckpt-{arch}` setiap ~2 jam dan di akhir sesi. Tidak pernah memblokir training kalau
Kaggle API belum terkonfigurasi di sesi berjalan.


In [ ]:
_last_push_ts = time.time()


def maybe_push_checkpoints_to_kaggle(cfg, force=False, interval_s=2 * 3600):
    global _last_push_ts
    if not force and (time.time() - _last_push_ts) < interval_s:
        return
    try:
        import subprocess
        slug = cfg["CHECKPOINT_DATASET_SLUG"]
        meta = {"title": slug, "id": f"{os.environ.get('KAGGLE_USERNAME', '<username>')}/{slug}",
                "licenses": [{"name": "CC0-1.0"}]}
        with open(CKPT_ROOT / "dataset-metadata.json", "w") as f:
            json.dump(meta, f, indent=2)
        subprocess.run(["kaggle", "datasets", "version", "-p", str(CKPT_ROOT), "-m",
                         f"checkpoint push @ {time.time():.0f}", "-r", "zip"], check=False)
        _last_push_ts = time.time()
    except Exception as e:
        print(f"push checkpoint dilewati (non-fatal): {e}")


## §7.8 Ekspor OOF (di akhir training tiap fold)

Load `best.ckpt` (sesuai variant bobot yang tercatat), jalankan EVAL transform + TTA persis sama
dengan yang dipakai di NB04, tulis `oof_{arch}_fold{k}.csv` sesuai skema kanonis (PRD §5.3).


In [ ]:
def generate_oof_for_fold(fold: int, cfg: dict):
    oof_path = OOF_ROOT / f"oof_{cfg['ARCH_NAME']}_fold{fold}.csv"
    if oof_path.exists():
        print(f"OOF fold {fold} sudah ada — dilewati")
        return

    best = torch.load(fold_dir(fold) / "best.ckpt", map_location="cpu")
    img_size = best["img_size"]
    model = create_model(cfg["TIMM_CHECKPOINT_TAG"], cfg["DROP_PATH_RATE"], img_size).to(DEVICE)
    model.load_state_dict(best["weights"])
    model.eval()
    mean, std = resolve_norm_stats(model)

    fa = pd.read_csv(FOLD_ASSIGNMENT_PATH)
    val_rows = fa[fa["fold"] == fold]
    eval_tf = build_eval_transform(img_size, mean, std)
    val_ds = WasteDataset(val_rows["image_id"].values, val_rows["label"].values, eval_tf)
    val_loader = make_loader(val_ds, shuffle=False, batch_size=32, drop_last=False)

    rows = []
    idx = 0
    label_lookup = dict(zip(val_rows["image_id"].values, val_rows["label"].values))
    for images, _labels in tqdm(val_loader, desc=f"OOF fold{fold}", leave=False):
        probs = tta_predict_probs(model, images).cpu().numpy()
        batch_ids = val_ds.image_ids[idx: idx + len(images)]
        idx += len(images)
        for image_id, p in zip(batch_ids, probs):
            rows.append({
                "image_id": int(image_id), "prob_0": float(p[0]), "prob_1": float(p[1]), "prob_2": float(p[2]),
                "true_label": int(label_lookup[int(image_id)]), "fold": fold,
            })

    oof_df = pd.DataFrame(rows).sort_values("image_id").reset_index(drop=True)
    oof_df.to_csv(oof_path, index=False)

    sidecar = {**PROVENANCE, "arch": cfg["ARCH_NAME"], "fold": fold,
               "weight_variant": best["variant"], "img_size": img_size, "tta_spec_version": TTA_SPEC_VERSION}
    with open(OOF_ROOT / f"oof_{cfg['ARCH_NAME']}_fold{fold}.json", "w") as f:
        json.dump(sidecar, f, indent=2)
    print(f"menulis {oof_path} ({len(oof_df)} baris)")


## G0 — Smoke test (setiap anggota, <30 menit GPU, SEBELUM run panjang mana pun)

Gate resolusi §7.7 sudah jalan di atas. Tambahan di sini: run overfit 50-step di subset kecil
(train loss harus turun) dan drill kill/resume checkpoint (kill kernel di tengah epoch, restart,
verifikasi resume lanjut dengan benar termasuk `epochs_since_improvement`). Jalankan sel ini
manual sekali per sesi sebelum commit ke fold penuh; bukan bagian dari loop fold otomatis.


In [ ]:
def g0_tiny_overfit_check(cfg, n_samples=64, steps=50):
    fa = pd.read_csv(FOLD_ASSIGNMENT_PATH)
    tiny = fa.sample(n=n_samples, random_state=SEED)
    img_size = cfg["STAGE_RESOLUTIONS"][1]
    model = create_model(cfg["TIMM_CHECKPOINT_TAG"], cfg["DROP_PATH_RATE"], img_size).to(DEVICE)
    mean, std = resolve_norm_stats(model)
    tf = build_train_transform(img_size, mean, std, erasing_p=0.0, color_jitter_scale=0.0)
    ds = WasteDataset(tiny["image_id"].values, tiny["label"].values, tf)
    loader = make_loader(ds, shuffle=True, batch_size=8, drop_last=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    losses = []
    model.train()
    step = 0
    while step < steps:
        for images, targets in loader:
            if step >= steps:
                break
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(images), targets)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            step += 1
    print(f"G0 tiny-overfit: rata-rata loss[0:5]={np.mean(losses[:5]):.4f} -> loss[-5:]={np.mean(losses[-5:]):.4f}")
    assert np.mean(losses[-5:]) < np.mean(losses[:5]), "train loss tidak turun di subset kecil — investigasi dulu sebelum run panjang"
    print("G0 tiny-overfit check LOLOS")


# Jalankan manual sebelum Gate G1 / run panjang mana pun:
# g0_tiny_overfit_check(CONFIG)
# (drill kill/resume: interrupt loop fold di tengah epoch, restart notebook, konfirmasi run_fold()
#  resume dari last.ckpt dengan epochs_since_improvement tetap terjaga — cek kontinuitas training_log.csv.)


## Eksekusi utama — loop 5 fold

Gate rule (PRD §3): anggota 2-4 baru mulai run berat setelah baseline Gate G1 milik Lead
(ConvNeXt V2, fold 0, 3 stage + OOF) lolos.


In [ ]:
for _fold in range(5):
    run_fold(_fold, CONFIG)
    generate_oof_for_fold(_fold, CONFIG)
    maybe_push_checkpoints_to_kaggle(CONFIG)

maybe_push_checkpoints_to_kaggle(CONFIG, force=True)
print(f"{CONFIG['ARCH_NAME']}: kelima fold selesai training dan OOF sudah diekspor.")
